# Day 04：自回归循环的浪费

对应 [`docs/day03-04.md`](../docs/day03-04.md) 的**任务 8**，是 Day 4 最重要的产出。

目的是把「无 KV Cache 有多浪费」**测出来**，为学习单元 Day 26～30 实现 Cache 留一个 baseline。

注意今天这个模型**没有 Attention**，所以重复计算的只是 embedding 查表和一个 GEMM。
等 Attention 加上之后浪费会更严重，因为 Attention 的中间矩阵随 `S²` 增长。

## 1. 输入长度单调递增

在 `generate_greedy` 的循环里打印每轮的 `input_ids.shape[1]`：

```text
第 1 轮 forward 10 个 token → 产出第 11 个
第 2 轮 forward 11 个 token → 产出第 12 个   ← 前 10 个又算了一遍
第 3 轮 forward 12 个 token → 产出第 13 个   ← 前 11 个又算了一遍
```

In [ ]:
import torch
from torch import nn

from mini_transformer.generate import generate_greedy


class AlwaysTokenModel(nn.Module):
    """每轮都选择同一个 token；只用来观察生成循环，不代表真实语言模型。"""

    def __init__(self, vocab_size: int, token_id: int):
        super().__init__()
        self.vocab_size = vocab_size
        self.token_id = token_id

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        batch_size, sequence_length = input_ids.shape
        print(f"  model 收到 input_ids，shape={tuple(input_ids.shape)}")
        logits = torch.zeros(batch_size, sequence_length, self.vocab_size)
        logits[..., self.token_id] = 1.0
        return logits


prompt_ids = torch.tensor([[10, 11, 12, 13]])
output_ids = generate_greedy(
    AlwaysTokenModel(vocab_size=32, token_id=7),
    prompt_ids,
    max_new_tokens=3,
)

print("\nprompt:", prompt_ids.tolist())
print("output:", output_ids.tolist())

# 观察：模型依次收到了长度 4、5、6 的序列；前缀每一轮都重新进入 forward。
assert output_ids.tolist() == [[10, 11, 12, 13, 7, 7, 7]]


## 2. 重算的结果每轮完全相同

这是「浪费」成立的前提：如果重算的结果不同，那就不算浪费。

把第 1 轮和第 2 轮**前 10 个位置**的 hidden_states 抓出来对比，
应当逐元素相等（用 `torch.testing.assert_close`）。

In [ ]:
import torch

from mini_transformer.tiny_lm import TinyLM

torch.manual_seed(0)
model = TinyLM(vocab_size=32, hidden_size=8)

first_input = torch.tensor([[10, 11, 12, 13]])
second_input = torch.tensor([[10, 11, 12, 13, 7]])

# 今天的 TinyLM 没有 Attention；这里的 hidden_states 就是 embedding 查表结果。
# 未来加了 Attention 后，因果性仍保证旧位置的结果不该因「在末尾追加新 token」而变化。
first_hidden = model.embedding(first_input)
second_hidden = model.embedding(second_input)

print("第 1 轮 hidden_states:", tuple(first_hidden.shape))
print("第 2 轮 hidden_states:", tuple(second_hidden.shape))
torch.testing.assert_close(first_hidden, second_hidden[:, : first_input.size(1), :])
print("前缀 4 个位置逐元素一致：True")


## 3. 累计计算量对比

累加总共处理了多少个 token-位置，和「理想情况」（有 Cache）对比。

公式：无 Cache 是 `Σ(P .. P+N-1)`，有 Cache 是 `P + (N-1)`。

**自己动手算这两行，别只是抄表**：

| prompt 长度 | 生成数 | 无 Cache 累计 | 有 Cache 累计 | 倍数 |
|---|---|---|---|---|
| 10 | 20 | 390 | 29 | 13.4× |
| 1024 | 100 | 107,350 | 1,123 | 95.6× |

In [ ]:
def token_positions_processed(prompt_length: int, num_new_tokens: int) -> tuple[int, int]:
    """返回无 Cache 与理想有 Cache 时，模型累计处理的 token-位置数。"""
    if prompt_length <= 0:
        raise ValueError("prompt_length must be positive")
    if num_new_tokens <= 0:
        raise ValueError("num_new_tokens must be positive")

    # 无 Cache：第 k 轮重新处理 P + k 个 token，k = 0 .. N-1
    no_cache = sum(range(prompt_length, prompt_length + num_new_tokens))
    # 有 Cache：prefill 处理 P 个；后续 N-1 个 decode 步各只处理新 token
    with_cache = prompt_length + num_new_tokens - 1
    return no_cache, with_cache


print(f"{'P':>6} {'N':>6} {'无 Cache':>12} {'有 Cache':>12} {'倍数':>8}")
print("-" * 52)
for prompt_length, num_new_tokens in [(10, 20), (1024, 100), (10, 1), (100, 100)]:
    no_cache, with_cache = token_positions_processed(prompt_length, num_new_tokens)
    print(f"{prompt_length:>6} {num_new_tokens:>6} {no_cache:>12,} {with_cache:>12,} {no_cache / with_cache:>7.1f}x")

assert token_positions_processed(10, 20) == (390, 29)
assert token_positions_processed(1024, 100) == (107_350, 1_123)


## 4. 结论

把结论写进 `docs/concepts/03-autoregressive-loop.md`（任务 6 新建）。

一句话版本：**这个浪费就是 KV Cache 存在的全部理由。**